# 📘 Intelligent Mentoring System Boilerplate
## Using Mehyaar/Annotated_NER_PDF_Resumes Dataset

In [4]:
!pip install resume-parser --quiet
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 5.5 MB/s  0:00:02 eta 0:00:01

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [5]:
import json
import os

print("Loading resumes...")

try:
    directory_path = "data/ResumesJsonAnnotated"

    data = []
    for filename in os.listdir(directory_path):
        if filename.endswith(".json"):
            with open(os.path.join(directory_path, filename), "r") as file:
                data.append(json.load(file))

    print(f"Loaded {len(data)} resumes from the dataset")

except Exception as e:
    print(f"Error: {e}")
    import traceback
    traceback.print_exc()


Loading resumes...
Loaded 5029 resumes from the dataset


In [6]:
import tempfile
import os
import logging
from resume_parser import resumeparse

logging.getLogger().setLevel(logging.CRITICAL)

successful_parses = 0
failed_parses = 0
error_types = {}

def clean_unicode_text(text):
    if not isinstance(text, str):
        text = str(text)
    cleaned = text.encode('utf-8', errors='ignore').decode('utf-8')
    cleaned = cleaned.replace('\\ud83d', '').replace('\\udcxx', '')
    return cleaned

def safe_parse_resume(text, idx):
    temp_file_path = None
    try:
        clean_text = clean_unicode_text(text)
        with tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8', errors='ignore') as temp_file:
            temp_file.write(clean_text)
            temp_file_path = temp_file.name

        parsed = resumeparse.read_file(temp_file_path)
        if not isinstance(parsed, dict):
            raise ValueError("Parser returned invalid data type")

        cleaned_parsed = {}
        for key, value in parsed.items():
            if value is not None:
                if isinstance(value, list):
                    cleaned_list = []
                    for v in value:
                        if v is not None:
                            cleaned_list.append(clean_unicode_text(v) if isinstance(v, str) else v)
                    cleaned_parsed[key] = cleaned_list
                elif isinstance(value, str):
                    cleaned_parsed[key] = clean_unicode_text(value)
                else:
                    cleaned_parsed[key] = value

        return cleaned_parsed

    except Exception as e:
        print(f"Error parsing resume at index {idx}: {e}")
        error_key = f"{type(e).__name__}: {str(e)[:50]}"
        error_types[error_key] = error_types.get(error_key, 0) + 1
        return None

    finally:
        if temp_file_path and os.path.exists(temp_file_path):
            try:
                os.unlink(temp_file_path)
            except:
                pass

for idx, resume_data in enumerate(data):
    parsed = safe_parse_resume(resume_data['text'], idx)
    if parsed is not None:
        resume_data['parsed'] = parsed
        successful_parses += 1
    else:
        resume_data['parsed'] = None
        failed_parses += 1

    if (idx + 1) % 100 == 0:
        print(f"Processed {idx + 1}/{len(data)} resumes. Success: {successful_parses}, Failed: {failed_parses}")

print("Results:")
print(f"Successful: {successful_parses}")
print(f"Failed: {failed_parses}")
print(f"Success rate: {(successful_parses/len(data)*100):.1f}%")

if error_types:
    print("Top error types:")
    for error_key, count in sorted(error_types.items(), key=lambda x: x[1], reverse=True)[:5]:
        print(f"{error_key}: {count}")

logging.getLogger().setLevel(logging.WARNING)
print("Processing complete.")


Processed 100/5029 resumes. Success: 100, Failed: 0
Processed 200/5029 resumes. Success: 200, Failed: 0
Processed 300/5029 resumes. Success: 300, Failed: 0
Processed 400/5029 resumes. Success: 400, Failed: 0
Processed 500/5029 resumes. Success: 500, Failed: 0
Processed 600/5029 resumes. Success: 600, Failed: 0
Processed 700/5029 resumes. Success: 700, Failed: 0
Processed 800/5029 resumes. Success: 800, Failed: 0
Processed 900/5029 resumes. Success: 900, Failed: 0
Processed 1000/5029 resumes. Success: 1000, Failed: 0
Processed 1100/5029 resumes. Success: 1100, Failed: 0
Processed 1200/5029 resumes. Success: 1200, Failed: 0
Processed 1300/5029 resumes. Success: 1300, Failed: 0
Processed 1400/5029 resumes. Success: 1400, Failed: 0
Processed 1500/5029 resumes. Success: 1500, Failed: 0
Processed 1600/5029 resumes. Success: 1600, Failed: 0
Processed 1700/5029 resumes. Success: 1700, Failed: 0
Processed 1800/5029 resumes. Success: 1800, Failed: 0
Processed 1900/5029 resumes. Success: 1900, Fa

In [7]:
import json

# Save the data to a JSON file
with open('parsed_resumes.json', 'w') as f:
    json.dump(data, f)
    
print(f"Saved {len(data)} resumes to parsed_resumes.json")

Saved 5029 resumes to parsed_resumes.json
